## >>> VERSION: 2026-07-25  ·  rcdnet-alone-v1  <<<
**RCDNet alone on the full 500-image benchmark (INSTRUMENTED).**

Runs RCDNet (SPA-Data weights) on all 500 images in ONE pass. RCDNet is fast
(~30 min for 500), well under Kaggle's kernel recycle, so **no batching is needed** —
attach the whole 500 dataset and run top to bottom.

The eval line is IDENTICAL to the working 150-study RCDNet notebook (the `!` shell
magic invocation of `rcdnet_cascade.py`); only metrics/stats cells are added around it.

All metrics at a fixed **256x256** (same scorer/resolution as the DiT-alone run, so the
Stage 3 paired tests are valid).

**Attach as inputs:** RaindropClarity code, the rcdnet code (with `spa_model_best.pt`
+ `init_kernel.mat`), the 500 dataset (`Drop/` + `Clear/`), and `score_pairs.py`.
**GPU T4 x2, Internet on.**

**After it finishes:** download `metrics.zip` (CSVs + stats) and `rcdnet_outputs.zip`
(the 500 restored images).

In [ ]:
# --- setup: copy RaindropClarity (for utils + score_pairs) and the rcdnet code ---
import os, shutil, glob
src = None
for d, _, files in os.walk('/kaggle/input'):
    if 'eval_diffusion_day_dit.py' in files:
        src = d; break
assert src, 'Could not find the RaindropClarity code under /kaggle/input'
shutil.copytree(src, '/kaggle/working/RaindropClarity', dirs_exist_ok=True)

rcd = None
for d, _, files in os.walk('/kaggle/input'):
    if 'rcdnet_cascade.py' in files:
        rcd = d; break
assert rcd, 'Could not find rcdnet_cascade.py under /kaggle/input'
shutil.copytree(rcd, '/kaggle/working/rcdnet', dirs_exist_ok=True)

os.chdir('/kaggle/working/RaindropClarity')
print('cwd    :', os.getcwd())
print('rcdnet :', rcd)
for s in glob.glob('/kaggle/input/**/score_pairs.py', recursive=True):
    shutil.copy(s, 'score_pairs.py'); print('got', s)
os.makedirs('/kaggle/working/metrics', exist_ok=True)

In [ ]:
!pip install lpips -q

In [ ]:
# --- locate the 500 dataset (Drop/ + Clear/) and the SPA/RCDNet checkpoint ---
import os, glob, torch
DATA = None
for d, subs, _ in os.walk('/kaggle/input'):
    if 'Drop' in subs and 'Clear' in subs:
        DATA = d; break
assert DATA, "Couldn't find Drop/ and Clear/ under /kaggle/input"
spa = glob.glob('/kaggle/input/**/spa_model_best.pt', recursive=True)
if not spa:
    spa = glob.glob('/kaggle/working/**/spa_model_best.pt', recursive=True)
assert spa, "Couldn't find spa_model_best.pt (attach the rcdnet code dataset)"
os.environ['DATA'] = DATA
os.environ['SPA']  = spa[0]
n = len(glob.glob(os.path.join(DATA, 'Drop', '**', '*.png'), recursive=True))
print('DATA   :', DATA)
print('SPA    :', spa[0])
print('images :', n, '(expected 500)')
print('GPU    :', torch.cuda.is_available(), ' device_count:', torch.cuda.device_count())

## Plumbing smoke test (20 images) — confirms score_pairs + LPIPS + CSV work

In [ ]:
# 20-image smoke test on Drop vs Clear: verifies scoring plumbing in ~1 min before the run
!python score_pairs.py --pred "$DATA/Drop" --gt "$DATA/Clear" \
    --name _smoketest --out_dir /kaggle/working/metrics --limit 20
import pandas as pd
df = pd.read_csv('/kaggle/working/metrics/_smoketest_per_image.csv')
print('\nsmoke test rows:', len(df), '(expect 20)')
assert len(df) == 20, 'smoke test did not produce 20 rows - fix before continuing'
print('PLUMBING OK')

## Static resource stats — parameter count + model size (no GPU)

In [ ]:
# param count + checkpoint size, read robustly straight from spa_model_best.pt
# (handles a state_dict, a wrapped dict, OR a saved nn.Module - never crashes the run)
import torch, os, csv, glob
spa = os.environ['SPA']
size_mb = os.path.getsize(spa) / (1024 * 1024)
ck = torch.load(spa, map_location='cpu', weights_only=False)
state = ck
if hasattr(state, 'state_dict'):                       # a full nn.Module was saved
    state = state.state_dict()
if isinstance(state, dict):
    for k in ('state_dict', 'model', 'net', 'params', 'G', 'generator'):
        if k in state and isinstance(state[k], dict):
            state = state[k]; break
n_params = sum(v.numel() for v in state.values() if torch.is_tensor(v)) if isinstance(state, dict) else -1
os.makedirs('/kaggle/working/metrics', exist_ok=True)
with open('/kaggle/working/metrics/resource_static.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['model', 'param_count', 'param_count_millions', 'model_size_MB'])
    w.writerow(['RCDNet', n_params, round(n_params / 1e6, 3), round(size_mb, 2)])
print(f'RCDNet: params={n_params:,} ({n_params/1e6:.2f}M)  size={size_mb:.2f} MB')
print(open('/kaggle/working/metrics/resource_static.csv').read())

## RCDNet alone (the ~30 min run) — timed, with peak-GPU logging

In [ ]:
# RCDNet run via the EXACT working ! invocation (unchanged from the 150 notebook).
# Python timing + an nvidia-smi poller wrap around it to capture time + peak GPU memory.
import subprocess, time, os, glob
mem_log = '/kaggle/working/metrics/gpu_mem_rcdnet.csv'
logf = open(mem_log, 'w')
poller = subprocess.Popen(
    ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits', '-l', '1'],
    stdout=logf)
t0 = time.time()
!cd /kaggle/working/rcdnet && python rcdnet_cascade.py --in_dir "$DATA" --out_dir /kaggle/working/rcd_alone --model spa_model_best.pt
elapsed = time.time() - t0
poller.terminate(); logf.close()
n_out = len(glob.glob('/kaggle/working/rcd_alone/Drop/**/*.png', recursive=True))
print('RCDNet done. elapsed:', round(elapsed, 1), 's | images restored:', n_out)
os.environ['RCD_ELAPSED'] = str(elapsed)
os.environ['RCD_N'] = str(max(n_out, 1))

In [ ]:
# runtime resource stats -> resource_runtime.csv  (real full-500 total, not projected)
import csv as _csv, os
elapsed = float(os.environ['RCD_ELAPSED'])
n = int(os.environ['RCD_N'])
mem_vals = [int(l.strip()) for l in open('/kaggle/working/metrics/gpu_mem_rcdnet.csv') if l.strip().isdigit()]
peak = max(mem_vals) if mem_vals else -1
per_image = elapsed / n
row = {'model': 'RCDNet',
       'per_image_s': round(per_image, 4),
       'throughput_img_per_s': round(1.0 / per_image, 4),
       'total_time_s': round(elapsed, 2),
       'peak_gpu_mem_MB': peak,
       'n_images': n}
rt = '/kaggle/working/metrics/resource_runtime.csv'
with open(rt, 'w', newline='') as f:
    w = _csv.DictWriter(f, fieldnames=list(row.keys()))
    w.writeheader(); w.writerow(row)
print(row)
print(open(rt).read())

In [ ]:
# score RCDNet outputs at 256x256 -> rcdnet_alone_per_image.csv + summary
# (runs from cwd=RaindropClarity so `utils` imports; pred=restored, gt=ground truth)
import os
os.system('python score_pairs.py --pred /kaggle/working/rcd_alone/Drop '
          '--gt /kaggle/working/rcd_alone/Clear --name rcdnet_alone '
          '--out_dir /kaggle/working/metrics')
print(open('/kaggle/working/metrics/rcdnet_alone_summary.txt').read())

In [ ]:
# collect the 500 restored images into rcdnet_outputs/ and zip
import shutil, os, glob
shutil.rmtree('/kaggle/working/rcdnet_outputs', ignore_errors=True)
shutil.copytree('/kaggle/working/rcd_alone/Drop', '/kaggle/working/rcdnet_outputs')
os.system('cd /kaggle/working && rm -f rcdnet_outputs.zip && zip -r rcdnet_outputs.zip rcdnet_outputs -q')
n = len(glob.glob('/kaggle/working/rcdnet_outputs/**/*.png', recursive=True))
print(f'wrote rcdnet_outputs.zip ({n} images)')

## Package + verify

In [ ]:
import os
os.system('cd /kaggle/working && rm -f metrics.zip && zip -r metrics.zip metrics -q')
print('wrote metrics.zip')
print('DOWNLOAD from Output panel: metrics.zip (CSVs + stats), rcdnet_outputs.zip (restored images)')

In [ ]:
# final verification - report what exists and row counts (never crashes)
import os
try:
    import pandas as pd
except Exception:
    pd = None
m = '/kaggle/working/metrics'
def chk(p, expect=None):
    ok = os.path.exists(p); extra = ''
    if ok and p.endswith('.csv') and pd is not None:
        try:
            n = len(pd.read_csv(p)); extra = ' (%d rows)' % n
            if expect and n != expect: extra += '  << expected %d' % expect
        except Exception as e:
            extra = ' (unreadable: %s)' % e
    print(('OK   ' if ok else 'MISS ') + p + extra)
chk(m + '/rcdnet_alone_per_image.csv', 500)
chk(m + '/rcdnet_alone_summary.txt')
chk(m + '/resource_static.csv')
chk(m + '/resource_runtime.csv')
print('\nrcdnet_alone_per_image.csv should have 500 rows. All metrics at 256x256 (matches DiT alone).')